# CoRe-TFM: JMLR Robustness Experiments — Updated

This notebook is the current execution entry point for the reliability-aware robustness work.

**Important safeguards**
- The frozen benchmark at `results/q1_fast_complete_256_v1` is read-only.
- The notebook checks out the latest `main` branch.
- Derived analysis files are regenerated from scratch, so stale/empty CSVs cannot survive from an earlier run.
- Fresh robustness experiments remain separate from the frozen evidence and must not be reported as completed until their completion gates pass.


## 1. Colab setup and latest repository checkout

Use a GPU runtime. If fresh TabPFN-3 inference is enabled, add a Colab secret named `TABPFN_TOKEN`.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time
import pandas as pd
import numpy as np

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    ROOT = Path('/content/core-tfm')
    DRIVE_BASE = Path('/content/drive/MyDrive/CoRe_TFM_Q1')
else:
    ROOT = Path.cwd()
    DRIVE_BASE = ROOT / 'results'

BRANCH = 'main'

if not (ROOT / '.git').exists():
    subprocess.run(
        ['git','clone','https://github.com/bnssaanirudh/core-tfm.git',str(ROOT)],
        check=True
    )

subprocess.run(['git','fetch','origin'], cwd=ROOT, check=True)
subprocess.run(['git','checkout',BRANCH], cwd=ROOT, check=True)
subprocess.run(['git','reset','--hard',f'origin/{BRANCH}'], cwd=ROOT, check=True)

HEAD = subprocess.check_output(['git','rev-parse','HEAD'], cwd=ROOT, text=True).strip()
print('Repository:', ROOT)
print('Branch:', BRANCH)
print('Commit:', HEAD)

subprocess.run(
    [sys.executable,'-m','pip','install','-q','-e','.[test]',
     'pyyaml','ucimlrepo','tabicl==2.1.1','tabpfn','catboost==1.2.10'],
    cwd=ROOT, check=True
)

sys.path.insert(0, str(ROOT/'src'))
os.chdir(ROOT)


## 2. Reset derived outputs and audit the frozen evidence

This deliberately removes only `results/reliability_aware_v1`. It does **not** modify the frozen Q1 evidence.


In [ ]:
FROZEN = ROOT/'results'/'q1_fast_complete_256_v1'
DERIVED = ROOT/'results'/'reliability_aware_v1'

if DERIVED.exists():
    shutil.rmtree(DERIVED)
DERIVED.mkdir(parents=True, exist_ok=True)

AUDIT_OUT = DERIVED/'evidence_audit.json'
subprocess.run([
    sys.executable,
    str(ROOT/'experiments'/'audit_evidence_package.py'),
    '--results', str(FROZEN),
    '--output', str(AUDIT_OUT)
], check=True)

audit = json.loads(AUDIT_OUT.read_text())
print('Evidence audit completed.')
print('Rows:', audit.get('row_count'))
print('Expected rows:', audit.get('expected_row_count'))
print('Checksum mismatches:', audit.get('checksums', {}).get('mismatches'))
audit


## 3. Regenerate archive-derived reliability analyses

The fixed generator resolves the frozen benchmark's canonical method names (`selective_core`, `hard_core`, `soft_core_lambda_1`) and refuses to write empty CSVs.


In [ ]:
subprocess.run([
    sys.executable,
    str(ROOT/'experiments'/'run_reliability_aware_suite.py'),
    '--fold-results', str(FROZEN/'fold_results.csv'),
    '--output', str(DERIVED)
], check=True)

expected = [
    'oracle_selection_decomposition.csv',
    'inconsistency_vs_gain.csv',
    'model_view_reliability_proxy.csv',
    'candidate_family_headroom.csv',
    'model_level_policy_transfer_proxy.csv',
    'RUN_METADATA.json',
]
for name in expected:
    p = DERIVED/name
    assert p.exists(), f'Missing derived output: {p}'
    assert p.stat().st_size > 0, f'Empty derived output: {p}'
    print(f'{name}: {p.stat().st_size:,} bytes')

meta = json.loads((DERIVED/'RUN_METADATA.json').read_text())
print('Resolved methods:', meta.get('resolved_methods'))
print('Oracle rows:', meta.get('oracle_rows'))


## 4. Oracle opportunity versus selection regret — fixed cell

This cell now validates the file before calling `pd.read_csv`, so an empty or missing result produces a clear error.


In [ ]:
oracle_file = DERIVED/'oracle_selection_decomposition.csv'

if not oracle_file.exists():
    raise FileNotFoundError(
        f'{oracle_file} does not exist. Rerun Section 3.'
    )
if oracle_file.stat().st_size == 0:
    raise RuntimeError(
        f'{oracle_file} is empty. Pull latest main and rerun Sections 2–3.'
    )

oracle = pd.read_csv(oracle_file)
required_cols = {
    'dataset','model','fold','available_opportunity',
    'selection_regret','selective_minus_arithmetic'
}
missing_cols = required_cols - set(oracle.columns)
if missing_cols:
    raise RuntimeError(f'Oracle file is missing columns: {sorted(missing_cols)}')
if oracle.empty:
    raise RuntimeError('Oracle decomposition parsed successfully but has zero rows.')

print('Oracle shape:', oracle.shape)
display(
    oracle.groupby('model')[
        ['available_opportunity','selection_regret','selective_minus_arithmetic']
    ].mean()
)
display(oracle.head())


## 5. Does inconsistency magnitude predict reconciliation benefit?


In [ ]:
ig = pd.read_csv(DERIVED/'inconsistency_vs_gain.csv')
corr = json.loads((DERIVED/'inconsistency_vs_gain_correlations.json').read_text())
display(pd.DataFrame(corr).T)

ax = ig.plot.scatter(
    x='factorization_tv',
    y='selective_gain',
    title='Factorization inconsistency vs Selective-CoRe gain'
)
ax.axhline(0, linewidth=1)


## 6. Model-specific reliability proxy

This is archive-derived evidence. It is a proxy because the old benchmark did not save every direct-marginal metric needed for a complete four-view reliability decomposition.


In [ ]:
rel = pd.read_csv(DERIVED/'model_view_reliability_proxy.csv')
display(
    rel.groupby('model').agg({
        'j1_joint_nll':'mean',
        'j2_joint_nll':'mean',
        'j1_conditional_nll_a_given_b':'mean',
        'j2_conditional_nll_b_given_a':'mean',
        'raw_direction_gap':'mean',
        'factorization_tv':'mean',
    })
)


## 7. Freeze the new robustness protocol before new test scores

Do not change these settings after inspecting comparative test outcomes without creating a dated protocol amendment.


In [ ]:
import yaml

cfg = yaml.safe_load(
    (ROOT/'configs'/'reliability_aware_experiments.yaml').read_text()
)

ROBUST_RUN_ID = cfg.get('robustness_run', {}).get(
    'run_id', 'core_tfm_jmlr_robustness_v1'
)
ROBUST = DRIVE_BASE/ROBUST_RUN_ID
ROBUST.mkdir(parents=True, exist_ok=True)

protocol = {
    'run_id': ROBUST_RUN_ID,
    'source_branch': BRANCH,
    'source_commit': HEAD,
    'primary_models': cfg['models']['primary'],
    'datasets': [
        'anneal','car','credit','customer','diamonds',
        'marketing','mic','nursery','phishing','wine'
    ],
    'seed_robustness': cfg['seed_robustness'],
    'context_size': cfg['context_size'],
    'safe_selective': cfg['safe_selective'],
    'rare_class_sensitivity': cfg['rare_class_sensitivity'],
    'third_tfm': cfg['models']['third_tfm'],
    'outcome_blind_freeze': True,
}

protocol_path = ROBUST/'ROBUSTNESS_PROTOCOL.json'
if protocol_path.exists():
    previous = json.loads(protocol_path.read_text())
    print('Existing frozen protocol found. Not overwriting it automatically.')
    display(previous)
else:
    protocol_path.write_text(json.dumps(protocol, indent=2))
    print('Frozen:', protocol_path)
    display(protocol)


## 8. Multi-seed robustness plan

Required design: five constrained-sampling seeds × five outer folds × ten datasets × two primary TFMs. Each seed must generate genuinely new splits; do not relabel seed 42 outputs.

The current repo retains the original bounded benchmark notebook as the inference implementation reference. Fresh inference outputs belong under the new robustness run directory.


In [ ]:
SEEDS = cfg['seed_robustness']['seeds']
seed_plan = pd.DataFrame([
    {
        'seed': seed,
        'folds': cfg['seed_robustness']['folds'],
        'train_limit': cfg['seed_robustness']['train_limit'],
        'validation_limit': cfg['seed_robustness']['validation_limit'],
        'test_limit': cfg['seed_robustness']['test_limit'],
        'output': str(ROBUST/f'multi_seed/seed_{seed}')
    }
    for seed in SEEDS
])
(ROBUST/'multi_seed').mkdir(exist_ok=True)
seed_plan.to_csv(ROBUST/'multi_seed'/'PLAN.csv', index=False)
display(seed_plan)


## 9. Context-size sensitivity plan


In [ ]:
context_dir = ROBUST/'context_size'
context_dir.mkdir(exist_ok=True)

context_plan = pd.DataFrame([
    {
        'seed': seed,
        'train_size': n,
        'folds': cfg['context_size']['folds'],
        'models': ';'.join(cfg['models']['primary'])
    }
    for seed in cfg['context_size']['seeds']
    for n in cfg['context_size']['train_sizes']
])
context_plan.to_csv(context_dir/'PLAN.csv', index=False)
display(context_plan)


## 10. Rare-class robustness

Audited low-support datasets include Customer, Marketing, and Nursery. Thresholds and support-adaptive penalties must be chosen without using test outcomes.


In [ ]:
from core_tfm.research_extensions import support_adaptive_penalties

supports = pd.DataFrame({
    'dataset':['customer','marketing','nursery'],
    'minimum_support':[3,2,2]
})
for tau in cfg['rare_class_sensitivity']['support_adaptive_penalty']['taus']:
    supports[f'lambda_tau_{tau:g}'] = support_adaptive_penalties(
        supports.minimum_support.to_numpy(),
        base_lambda=10,
        tau=tau
    )
display(supports)


## 11. Safe Selective CoRe complexity schedule

All candidate scores used for selection must come from validation data. Test scores may be used only for final evaluation after the policy is frozen.


In [ ]:
from core_tfm.research_extensions import complexity_penalty

rows = []
for nv in cfg['safe_selective']['validation_sizes']:
    for m in [1,4,8,16,48]:
        rows.append({
            'n_validation': nv,
            'family_size': m,
            'penalty': complexity_penalty(nv, m)
        })
display(pd.DataFrame(rows))


## 12. Per-view reliability and inference-dispersion requirements

Fresh runs should archive direct marginal NLL/Brier/ECE, conditional NLL/Brier, per-class support, predictive entropy, per-example factorization TV, marginalization defects, and all validation candidate scores.

Repeated-inference Jensen–Shannon dispersion should be called **inference instability/predictive dispersion**, not epistemic uncertainty, unless a separate validation establishes that interpretation.


## 13. Third-TFM preflight

Do not add a third TFM only to increase model count. Require a released checkpoint/API, probability outputs, feasible conditional queries, compatible licensing, and practical bounded-context runtime. Keep `third_tfm: null` until a candidate passes this preflight.


## 14. Final evidence gate

A robustness result can enter the manuscript only after its own `COMPLETE.json`, environment metadata, failure log, checksums, and dataset-blocked statistical summary exist.


In [ ]:
required_groups = [
    'multi_seed',
    'context_size',
    'rare_class',
    'safe_selective',
    'view_reliability',
]
status = {
    group: (ROBUST/group/'COMPLETE.json').exists()
    for group in required_groups
}
print(status)
print('JMLR robustness gate:', 'PASS' if all(status.values()) else 'PENDING')
